ベース：なし（GeoTIFF軽量化処理）

人口ラスタ：WorldPop "Syrian Arab Republic 100m Population 2026"
https://hub.worldpop.org/geodata/summary?id=75632
syr_worldpop_2026.tif

分析手法：Rasterio Resampling（Average）

対象地域：
・Syria (National Scale)

その他：
・元ラスタ（EPSG:4326）のCRS・サイズ・範囲を確認。
・Resampling.average により、人口値を平均化しながら解像度を1/2へ縮小。
・Transformを再計算し、地理的位置情報を維持。
・軽量化後のGeoTIFFとして保存。
・保存後のCRS・幅・高さを再確認し、正常に出力されたことを検証。

In [1]:
# Imports
# Raster Resampleに必要なライブラリを読み込む

import rasterio
from rasterio.enums import Resampling

In [2]:
# ETL Extract
# 元の人口ラスタの情報を確認する

with rasterio.open("tif_1_syria_worldpop_2026.tif") as src:

    # Check
    # CRSを確認する
    print(src.crs)

    # Check
    # ラスタサイズを確認する
    print(src.width, src.height)

    # Check
    # 地理的範囲（Bounds）を確認する
    print(src.bounds)

EPSG:4326
7992 6011
BoundingBox(left=35.7166658038, bottom=32.310833540089995, right=42.37666577716, top=37.320000186719994)


In [3]:
# ETL Transform
# 人口ラスタを1/2サイズへリサンプリングする

scale_factor = 2

with rasterio.open("tif_1_syria_worldpop_2026.tif") as src:

    # Average Resamplingで人口値を平均化しながら縮小する
    data = src.read(
        1,
        out_shape=(
            src.height // scale_factor,
            src.width // scale_factor
        ),
        resampling=Resampling.average
    )

    # リサンプリング後のTransformを再計算する
    # ラスタサイズ変更後も地理的位置がズレないよう維持する
    transform = src.transform * src.transform.scale(
        (src.width / data.shape[-1]),
        (src.height / data.shape[-2])
    )

In [4]:
# Check
# リサンプリング後のデータサイズを確認する
print(data.shape)

(3005, 3996)


In [5]:
# Check
# 新しいTransformを確認する
print(transform)

| 0.00, 0.00, 35.72|
| 0.00,-0.00, 37.32|
| 0.00, 0.00, 1.00|


In [6]:
# ETL Load
# 軽量版GeoTIFFとして保存する

with rasterio.open("tif_1_syria_worldpop_2026.tif") as src:

    # 元ラスタのメタデータをコピーする
    profile = src.profile.copy()

    # 軽量化後の幅・高さ・Transformへ更新する
    profile.update(
        width=data.shape[1],
        height=data.shape[0],
        transform=transform
    )

    # 新しいGeoTIFFファイルを作成する
    with rasterio.open(
        "tif_4_syria_worldpop_resample.tif",
        "w",
        **profile
    ) as dst:

        # 軽量化後の人口ラスタを書き込む
        dst.write(data, 1)

print("Resample完了")

Resample完了


In [7]:
# Check
# 保存した軽量版GeoTIFFのCRS・サイズを確認する

with rasterio.open("tif_4_syria_worldpop_resample.tif") as src:

    print(src.crs)
    print(src.width)
    print(src.height)

EPSG:4326
3996
3005
